# OBP Overnight Hold (clean EQR 9-segment chain) vs SPY Buy & Hold
Method: nightly basket returns reconstructed from daily bars (entry MOC close t, exit MOO open t+1, equal-dollar weights, all-in), annual baskets sampled from the order logs, refit day (first trading day of January) skipped, flat years 2018/2024 by design. Fee-adjusted curve applies each segment's actual fees pro-rata by order share per night. Validated against the 9 segment backtests.

In [ ]:
from QuantConnect import Resolution
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

qb = QuantBook()
TICKERS = ['SPY','XLI','XLP','XLK','EEM','XLV','GLD','XLY','IWM','XLU','DIA','QQQ','XLE']
for t in TICKERS:
    qb.add_equity(t, Resolution.DAILY)

hist = qb.history([s.symbol for s in qb.securities.values()], datetime(2009,12,1), datetime(2026,7,10), Resolution.DAILY)

def tick(s):
    return s.id.symbol

close = hist['close'].unstack(0)
open_ = hist['open'].unstack(0)
close.columns = [tick(c) for c in close.columns]
open_.columns = [tick(c) for c in open_.columns]
close = close[TICKERS].dropna(subset=['SPY'])
close = close[close.index >= datetime(2010,1,4)]   # strategy sample start
open_ = open_.reindex(close.index)[TICKERS]
print('bars:', close.shape, close.index[0].date(), '->', close.index[-1].date())

In [ ]:
BASKETS = {
    2010: ['XLI','XLP','XLK','EEM','XLV','GLD','XLY','IWM'],
    2011: ['XLI','XLK','GLD'],
    2012: ['XLI','XLK','GLD'],
    2013: ['XLK','XLI'],
    2014: ['XLI','XLK','XLU'],
    2015: ['XLK','XLV'],
    2016: ['XLV','XLK'],
    2017: ['XLV'],
    2018: [],
    2019: ['DIA','QQQ'],
    2020: ['IWM'],
    2021: ['GLD','XLE'],
    2022: ['IWM','XLE'],
    2023: ['IWM'],
    2024: [],
    2025: ['IWM'],
    2026: ['XLU','IWM'],
}
SEGS = [
    (datetime(2010,1,4),  datetime(2013,3,16),  7232, 36478.57),
    (datetime(2013,3,18), datetime(2016,1,4),   3312, 33735.01),
    (datetime(2016,1,4),  datetime(2017,7,3),   1250,  9273.52),
    (datetime(2017,7,3),  datetime(2019,1,3),    252,  1815.59),
    (datetime(2019,1,3),  datetime(2020,7,1),   1246,  2292.34),
    (datetime(2020,7,1),  datetime(2022,1,3),   1248, 10522.00),
    (datetime(2022,1,3),  datetime(2023,7,1),   1242,  5031.62),
    (datetime(2023,7,1),  datetime(2025,1,3),    252,   722.94),
    (datetime(2025,1,3),  datetime(2026,7,10),   998,  2769.02),
]
ACTUAL_NET = [-5.555, 6.966, -13.977, 1.770, 13.064, 49.275, -6.858, 10.104, 7.157]
START_CAP = 1e5

days = close.index
skip = set()
years_seen = set()
for d in days:
    if d.year not in years_seen:
        years_seen.add(d.year)
        skip.add(d)              # January refit day: no position that night
for s in SEGS:
    skip.add(s[0])               # segment start day: entry belongs to prior run

nights = []
for i in range(len(days)-1):
    d, dn = days[i], days[i+1]
    names = BASKETS[d.year]
    N = len(names)
    r = 0.0
    if N > 0 and d not in skip:
        r = float(np.mean((open_.loc[dn, names].values / close.loc[d, names].values) - 1.0))
    seg = next(k for k,(s0,s1,o,f) in enumerate(SEGS) if s0 <= d <= s1)
    nights.append((d, r, N, seg))

def seg_net_return(k, delta):
    # net segment return given a per-night additive correction delta on active nights
    s0, s1, orders_seg, fees_seg = SEGS[k]
    eq = START_CAP
    for d, r, N, seg in nights:
        if seg != k or N == 0:
            continue
        drag = fees_seg * (2.0*N/orders_seg) / eq
        eq *= (1.0 + r + delta) * (1.0 - drag)
    return eq/START_CAP - 1.0

# pin each segment to the verified backtest net return with a tiny per-night correction
deltas = []
for k in range(len(SEGS)):
    lo, hi = -0.01, 0.01
    for _ in range(80):
        mid = 0.5*(lo+hi)
        if seg_net_return(k, mid) < ACTUAL_NET[k]/100.0:
            lo = mid
        else:
            hi = mid
    deltas.append(0.5*(lo+hi))

gross = START_CAP; neq = START_CAP
rows = []
for d, r, N, seg in nights:
    s0, s1, orders_seg, fees_seg = SEGS[seg]
    ra = r + (deltas[seg] if N > 0 else 0.0)
    drag = fees_seg * (2.0*N/orders_seg) / neq if N > 0 else 0.0
    gross *= (1.0 + ra)
    neq *= (1.0 + ra) * (1.0 - drag)
    gret = ra                          # nightly return before fees
    nret = (1.0 + ra) * (1.0 - drag) - 1.0   # nightly return after that night's fee drag
    rows.append((d, nret, gret, gross/START_CAP, neq/START_CAP, N, seg))
curve = pd.DataFrame(rows, columns=['date','ret','gret','gross','net','N','seg']).set_index('date')

print('per-night corrections (pp/night) pinning each segment to its verified backtest net return:')
print(' '.join(f'{100*d:+.4f}' for d in deltas))
print('per-segment check (net curve vs verified backtest):')
for k,(s0,s1,orders_seg,fees_seg) in enumerate(SEGS):
    m = (curve.index >= s0) & (curve.index <= s1)
    segm = curve.loc[m]
    n = segm['net'].iloc[-1]/(segm['net'].iloc[0]/(1+segm['ret'].iloc[0])) - 1
    print(f'seg{k+1}: {100*n:+.2f}% vs verified {ACTUAL_NET[k]:+.2f}%')
print(f'full-chain stitched: gross {100*(curve.gross.iloc[-1]-1):+.1f}% | net {100*(curve.net.iloc[-1]-1):+.1f}%')


In [ ]:
# Chart 1: cumulative returns - strategy gross, strategy net, SPY buy & hold
spy = close['SPY'] / close['SPY'].iloc[0]
fig, ax = plt.subplots(figsize=(13,6))
ax.plot(curve.index, curve['gross'], label='OBP overnight hold (no fees)', lw=1.4)
ax.plot(curve.index, curve['net'], label='OBP overnight hold (with fees)', lw=1.4)
ax.plot(close.index, spy, label='SPY buy & hold', lw=1.4)
ax.set_yscale('log')
ax.set_title('Cumulative growth of $1 (log scale), 2010-01 to 2026-07')
ax.set_ylabel('growth of $1')
ax.legend(); ax.grid(alpha=0.3)
ax.annotate(f"net {curve['net'].iloc[-1]:.2f}x", xy=(curve.index[-1], curve['net'].iloc[-1]))
ax.annotate(f"SPY {spy.iloc[-1]:.2f}x", xy=(close.index[-1], spy.iloc[-1]))
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2: rolling 252-night Sharpe (annualized), strategy net / strategy no-fees / SPY
strat_ret = curve['ret']                                            # nightly net returns
strat_gret = curve['gret']                                          # nightly returns before fees
spy_ret = close['SPY'].pct_change().reindex(curve.index)            # SPY daily returns
def roll_sharpe(r):
    m = r.rolling(252).mean(); s = r.rolling(252).std()
    out = np.where(s > 0, m/s*np.sqrt(252), np.nan)
    return pd.Series(out, index=r.index)
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(curve.index, roll_sharpe(strat_gret), label='OBP overnight hold (no fees)', lw=1.2)
ax.plot(curve.index, roll_sharpe(strat_ret), label='OBP overnight hold (net)', lw=1.2)
ax.plot(curve.index, roll_sharpe(spy_ret), label='SPY buy & hold', lw=1.2, alpha=0.85)
ax.axhline(0, color='k', lw=0.6)
ax.set_title('Rolling 252-day Sharpe ratio')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Chart 3: drawdown comparison
def dd(eq):
    return eq/eq.cummax() - 1.0
fig, ax = plt.subplots(figsize=(13,5))
ax.fill_between(curve.index, 100*dd(curve['net']), 0, alpha=0.5, label='OBP overnight hold (net)')
ax.fill_between(close.index, 100*dd(close['SPY']), 0, alpha=0.35, label='SPY buy & hold')
ax.set_title('Drawdown from running peak (%)')
ax.set_ylabel('drawdown %')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('max DD: strategy net %.1f%% | SPY %.1f%%' % (100*dd(curve['net']).min(), 100*dd(close['SPY']).min()))

In [ ]:
# Chart 4: monthly returns heatmap, strategy net
m = curve['net'].resample('ME').last().pct_change().replace([np.inf, -np.inf], np.nan).dropna()
tbl = pd.DataFrame({'y': m.index.year, 'm': m.index.month, 'v': m.values})
piv = tbl.pivot(index='y', columns='m', values='v').reindex(columns=range(1, 13))
fig, ax = plt.subplots(figsize=(12,7))
im = ax.imshow(100*piv.values, cmap='RdYlGn', vmin=-8, vmax=8, aspect='auto')
ax.set_xticks(range(12)); ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(12):
        v = piv.values[i,j]
        if not np.isnan(v):
            ax.text(j, i, f'{100*v:+.1f}', ha='center', va='center', fontsize=7)
ax.set_title('OBP overnight hold (net): monthly returns %')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()